# Lab 3: Generate, Check, and Store Data with PhyPhox

This lab introduces the full workflow for working with data collected by students using **PhyPhox**:
1. Generate data in the app.
2. Load the pointer to the recorded data file.
3. Extract recording metadata from the data export or the adjacent `meta/` folder.
4. Check the data quality in Python.
5. Store the cleaned dataset and a notebook-level summary in a structured way.

## Learning goals
- Understand how PhyPhox exports measurement files.
- Read exported data with Python.
- Distinguish between a pointer metadata file and metadata stored with the recorded data.
- Detect whether a table is Excel or one of the supported CSV formats.
- Extract metadata from XLS/XLSX workbooks or CSV sidecar `meta/` folders.
- Inspect and validate data quality.
- Save processed data so it can be reused later.

## Suggested workflow
- Students collect data using PhyPhox on their phones.
- They export the data as CSV or Excel.
- They set `metadata.json` to point to the recorded data file.
- They load the file, verify the detected format, and inspect extracted metadata.
- They save a clean version and a Lab 3 summary for later analysis.

In [ ]:
# Section 1: Import Required Libraries
from pathlib import Path
import csv
import json
import re

import pandas as pd
import numpy as np

print('Libraries imported successfully.')

## Section 2: Load Pointer Metadata and Extract Recorded Data Metadata

The central `metadata.json` file only points to the recorded data file. Metadata about the actual data collection must come from the data export itself:

- Excel/XLS/XLSX: metadata is treated as embedded in the workbook and is extracted from its sheets.
- CSV: the notebook checks whether a `meta/` folder exists next to the CSV file and loads metadata files from there.

The notebook also detects the table format: Excel, or CSV with comma/tabulator/semicolon separators and decimal point/comma.

In [ ]:
# Section 2: Load Pointer Metadata and Extract Recorded Data Metadata
project_root = Path.cwd()
metadata_path = project_root / 'metadata.json'

if metadata_path.exists():
    with metadata_path.open('r', encoding='utf-8') as f:
        metadata = json.load(f)
else:
    metadata = {'recorded_data_path': 'data/drivetrain/Example/Raw Data.csv'}

recorded_data_path = metadata.get('recorded_data_path')
if recorded_data_path is None:
    recorded_data_path = metadata.get('recorded_data', {}).get('path')
if recorded_data_path is None:
    recorded_data_path = metadata.get('recorded_data', {}).get('raw_data_file')
if recorded_data_path is None:
    raise ValueError('metadata.json must contain recorded_data_path.')

selected_data_path = project_root / recorded_data_path


def _read_text_sample(path, max_bytes=65536):
    raw = path.read_bytes()[:max_bytes]
    for encoding in ['utf-8-sig', 'utf-8', 'latin1']:
        try:
            return raw.decode(encoding), encoding
        except UnicodeDecodeError:
            pass
    return raw.decode('latin1', errors='replace'), 'latin1'


def detect_csv_format(path):
    sample, encoding = _read_text_sample(path)
    lines = [line for line in sample.splitlines() if line.strip()][:30]
    delimiters = [',', '\t', ';']
    delimiter_names = {',': 'comma', '\t': 'tabulator', ';': 'semicolon'}

    best = None
    for delimiter in delimiters:
        parsed = list(csv.reader(lines, delimiter=delimiter))
        widths = [len(row) for row in parsed if row]
        multi_column_rows = [width for width in widths if width > 1]
        if not multi_column_rows:
            score = 0
            common_width = 1
        else:
            common_width = max(set(multi_column_rows), key=multi_column_rows.count)
            score = multi_column_rows.count(common_width) * common_width
        candidate = (score, common_width, delimiter)
        if best is None or candidate > best:
            best = candidate

    delimiter = best[2]
    parsed = list(csv.reader(lines, delimiter=delimiter))
    data_tokens = []
    for row in parsed[1:]:
        data_tokens.extend([item.strip() for item in row])

    decimal_comma = sum(1 for item in data_tokens if re.fullmatch(r'[-+]?\d+,\d+(?:[eE][-+]?\d+)?', item))
    decimal_point = sum(1 for item in data_tokens if re.fullmatch(r'[-+]?\d+\.\d+(?:[eE][-+]?\d+)?', item))
    decimal = ',' if decimal_comma > decimal_point else '.'
    decimal_name = 'decimal comma' if decimal == ',' else 'decimal point'

    return {
        'container_format': 'csv',
        'format_label': f"csv ({delimiter_names[delimiter]}, {decimal_name})",
        'delimiter': delimiter,
        'decimal': decimal,
        'encoding': encoding
    }


def read_table(path, csv_format=None):
    suffix = path.suffix.lower()
    if suffix == '.csv':
        csv_format = csv_format or detect_csv_format(path)
        return pd.read_csv(
            path,
            sep=csv_format['delimiter'],
            decimal=csv_format['decimal'],
            encoding=csv_format['encoding']
        )
    if suffix in ['.xlsx', '.xls']:
        return pd.read_excel(path)
    raise ValueError(f'Unsupported file type: {suffix}')


def extract_csv_sidecar_metadata(data_path):
    meta_dir = data_path.parent / 'meta'
    result = {
        'source': 'csv_sidecar_meta_folder',
        'meta_folder': str(meta_dir.relative_to(project_root)).replace('\\', '/'),
        'exists': meta_dir.exists(),
        'files': {}
    }
    if not meta_dir.exists():
        return result

    for meta_file in sorted(meta_dir.glob('*')):
        if not meta_file.is_file() or meta_file.suffix.lower() not in ['.csv', '.xlsx', '.xls']:
            continue
        try:
            if meta_file.suffix.lower() == '.csv':
                fmt = detect_csv_format(meta_file)
                frame = read_table(meta_file, fmt)
            else:
                fmt = {'container_format': 'excel', 'format_label': 'excel'}
                frame = pd.read_excel(meta_file)
            result['files'][str(meta_file.relative_to(project_root)).replace('\\', '/')] = {
                'format': fmt,
                'columns': frame.columns.tolist(),
                'row_count': int(len(frame)),
                'records': frame.head(50).where(pd.notna(frame), None).to_dict(orient='records')
            }
        except Exception as error:
            result['files'][str(meta_file.relative_to(project_root)).replace('\\', '/')] = {
                'error': str(error)
            }
    return result


def extract_excel_embedded_metadata(data_path):
    result = {
        'source': 'excel_workbook',
        'workbook': str(data_path.relative_to(project_root)).replace('\\', '/'),
        'sheets': [],
        'sheet_previews': {}
    }
    excel_file = pd.ExcelFile(data_path)
    result['sheets'] = excel_file.sheet_names
    for sheet_name in excel_file.sheet_names:
        try:
            preview = pd.read_excel(data_path, sheet_name=sheet_name, nrows=30)
            result['sheet_previews'][sheet_name] = {
                'columns': preview.columns.tolist(),
                'row_count_preview': int(len(preview)),
                'records': preview.head(10).where(pd.notna(preview), None).to_dict(orient='records')
            }
        except Exception as error:
            result['sheet_previews'][sheet_name] = {'error': str(error)}
    return result


def extract_recorded_data_metadata(data_path):
    suffix = data_path.suffix.lower()
    if suffix == '.csv':
        fmt = detect_csv_format(data_path)
        extracted = extract_csv_sidecar_metadata(data_path)
    elif suffix in ['.xlsx', '.xls']:
        fmt = {'container_format': 'excel', 'format_label': 'excel'}
        extracted = extract_excel_embedded_metadata(data_path)
    else:
        fmt = {'container_format': suffix.lstrip('.') or 'unknown', 'format_label': 'unknown'}
        extracted = {'source': 'unsupported'}

    return {
        'recorded_data_path': str(data_path.relative_to(project_root)).replace('\\', '/'),
        'detected_format': fmt,
        'extracted_metadata': extracted
    }


recorded_data_metadata = extract_recorded_data_metadata(selected_data_path)

print('Pointer metadata file:', metadata_path)
print('Recorded data:', selected_data_path)
print('Detected format:', recorded_data_metadata['detected_format']['format_label'])
display(pd.json_normalize(recorded_data_metadata, sep='.').T.rename(columns={0: 'value'}))

## Section 3: Override Recorded Data Path if Needed

Use this cell only to switch to another recorded data file. The override changes the pointer to the data file, not the extracted collection metadata.

In [ ]:
# Section 3: Override Recorded Data Path if Needed
# The metadata file should only point to the recorded data file.
# Change this value if you want to use another measurement file.
recorded_data_path_override = None
# recorded_data_path_override = 'data/suspension/Example/Beschleunigung ohne g.xls'
# recorded_data_path_override = 'data/drivetrain/Example/Raw Data.csv'

save_path_override = False

if recorded_data_path_override:
    metadata['recorded_data_path'] = recorded_data_path_override
    selected_data_path = project_root / recorded_data_path_override
    recorded_data_metadata = extract_recorded_data_metadata(selected_data_path)

if save_path_override:
    with metadata_path.open('w', encoding='utf-8') as f:
        json.dump({'recorded_data_path': metadata['recorded_data_path']}, f, indent=2)
    print('Saved recorded data path override:', metadata_path)
else:
    print('Override applied for this notebook session only.' if recorded_data_path_override else 'No override applied.')

print('Recorded data:', selected_data_path)
print('Detected format:', recorded_data_metadata['detected_format']['format_label'])
display(pd.json_normalize(recorded_data_metadata, sep='.').T.rename(columns={0: 'value'}))

## Section 4: Set Up File Paths

The raw data folder is the folder that contains the recorded data file from `metadata.json`. Processed outputs are written to `data/processed`.

In [ ]:
# Section 4: Set Up File Paths
processed_data_folder = project_root / 'data' / 'processed'
raw_data_folder = selected_data_path.parent

raw_data_folder.mkdir(parents=True, exist_ok=True)
processed_data_folder.mkdir(parents=True, exist_ok=True)

print('Raw folder:', raw_data_folder)
print('Processed folder:', processed_data_folder)
print('Pointer metadata path:', metadata_path)
print('Recorded data path:', selected_data_path)

## Section 5: Read PhyPhox Export Files

In this section, students load the recorded data file into Python. The notebook uses the detected file format from Section 2, so CSV separators and decimal notation are handled explicitly.

In [ ]:
# Section 5: Read PhyPhox Export Files
raw_files = sorted([f for f in raw_data_folder.glob('*') if f.is_file()])

print('Found files:')
for file in raw_files:
    print(' -', file.name)

data_path = selected_data_path
if data_path.exists():
    df = read_table(data_path, recorded_data_metadata.get('detected_format'))
    print(f'\nLoaded file: {data_path.name}')
    print('Detected format:', recorded_data_metadata['detected_format']['format_label'])
    display(df.head())
else:
    print('Recorded data file not found:', data_path)

## Section 6: Inspect the Data Structure

Now that the file is loaded, inspect the table carefully.

Questions to ask:
- How many rows and columns does the dataset have?
- What do the column names mean?
- Are the measurement values in a sensible range?
- Does the time column look correct?

In [ ]:
# Section 6: Inspect the Data Structure
print('Shape:', df.shape)
print('\nColumns:')
print(df.columns.tolist())
print('\nData types:')
print(df.dtypes)
print('\nFirst few rows:')
display(df.head())
print('\nSummary statistics:')
display(df.describe(include='all').T)

## Section 7: Check Data Quality

This section focuses on finding problems before analysis.

Check for:
- Missing values
- Duplicate rows
- Strange outliers
- Unusual time jumps
- Units or labels that may be inconsistent

In [ ]:
# Section 7: Check Data Quality
print('Missing values per column:')
print(df.isna().sum())

print('\nDuplicate rows:', df.duplicated().sum())

# Quick numeric checks
numeric_columns = df.select_dtypes(include=[np.number]).columns
for col in numeric_columns:
    series = df[col]
    print(f'\nColumn: {col}')
    print(f'  Min: {series.min()}')
    print(f'  Max: {series.max()}')
    print(f'  Mean: {series.mean()}')

# Optional: show rows with missing values
if df.isna().any().any():
    display(df[df.isna().any(axis=1)].head())

## Section 8: Clean and Standardize the Data

Use this section to make the data easier to analyze.

Possible steps:
- Rename columns to clear names.
- Convert time values to a correct format.
- Remove duplicate rows.
- Keep only columns that you need for the next analysis.

In [ ]:
# Section 8: Clean and Standardize the Data
# Example cleanup steps; adapt these to your dataset.

# Remove exact duplicate rows if needed
df_clean = df.drop_duplicates().copy()

# Rename columns if you want clearer names
# df_clean = df_clean.rename(columns={
#     'Time (s)': 'time_s',
#     'Illuminance (lx)': 'illuminance_lx'
# })

# Convert a time column if present and if it looks numeric
if 'Time (s)' in df_clean.columns:
    df_clean['Time (s)'] = pd.to_numeric(df_clean['Time (s)'], errors='coerce')

# Keep a small set of relevant columns if needed
# df_clean = df_clean[['time_s', 'illuminance_lx']]

print('Cleaned dataset shape:', df_clean.shape)
display(df_clean.head())

## Section 9: Summarize Recorded Data for This Notebook

This section creates a notebook-level summary from the loaded data and the extracted recording metadata. It does not write collection metadata back into `metadata.json`; the central metadata file remains only a pointer to the recorded data file.

In [ ]:
# Section 9: Summarize Recorded Data for This Notebook
if 'df' in locals() and 'data_path' in locals():
    numeric_columns = df.select_dtypes(include=[np.number]).columns.tolist()
    time_candidates = [c for c in numeric_columns if 'time' in c.lower() or 'zeit' in c.lower()]
    time_column = time_candidates[0] if time_candidates else ''
    value_columns = [c for c in numeric_columns if c != time_column]

    duration_s = None
    if time_column:
        duration_s = float(df[time_column].max() - df[time_column].min())

    lab3_data_summary = {
        'recorded_data_path': str(data_path.relative_to(project_root)).replace('\\', '/'),
        'detected_format': recorded_data_metadata['detected_format'],
        'extracted_recording_metadata': recorded_data_metadata['extracted_metadata'],
        'columns': df.columns.tolist(),
        'time_column': time_column,
        'value_columns': value_columns,
        'row_count': int(df.shape[0]),
        'column_count': int(df.shape[1]),
        'duration_s': duration_s,
        'missing_values': {col: int(value) for col, value in df.isna().sum().items()},
        'duplicate_rows': int(df.duplicated().sum()),
        'numeric_summary': df[numeric_columns].describe().to_dict() if numeric_columns else {}
    }

    print('Data summary for this lab session:')
    display(pd.json_normalize(lab3_data_summary, sep='.').T.rename(columns={0: 'value'}))
else:
    print('No loaded dataset found yet. Run the data import section first.')

## Section 10: Store the Processed Data

Save the cleaned data and, if available, the Lab 3 summary. The summary can be reused for checking, but the source of recording metadata remains the data export itself.

In [ ]:
# Section 10: Store the Processed Data
from datetime import datetime

experiment_name = selected_data_path.stem.replace(' ', '_')
output_name = f'{experiment_name}_processed'

csv_output = processed_data_folder / f'{output_name}.csv'
json_output = processed_data_folder / f'{output_name}.json'
summary_output = processed_data_folder / f'{output_name}_lab3_summary.json'

# Save cleaned data and a Lab 3 summary. The central metadata file remains only a pointer.
if 'df_clean' in locals():
    df_clean.to_csv(csv_output, index=False)
    df_clean.to_json(json_output, orient='records', indent=2)
    if 'lab3_data_summary' in locals():
        with summary_output.open('w', encoding='utf-8') as f:
            json.dump(lab3_data_summary, f, indent=2)

    print('Saved CSV:', csv_output)
    print('Saved JSON:', json_output)
    if 'lab3_data_summary' in locals():
        print('Saved Lab 3 summary:', summary_output)
else:
    print('No cleaned dataset found yet. Run the cleaning section first.')

## Section 11: Verify Saved Outputs

This final step checks that the saved files really contain the intended data.

Run the next cell to read the saved CSV and JSON files back and compare them with the cleaned table.

In [ ]:
# Section 11: Verify Saved Outputs
if csv_output.exists():
    df_saved_csv = pd.read_csv(csv_output)
    print('CSV loaded successfully.')
    display(df_saved_csv.head())
    print('CSV shape:', df_saved_csv.shape)

if json_output.exists():
    with json_output.open('r', encoding='utf-8') as f:
        df_saved_json = json.load(f)
    print('\nJSON loaded successfully.')
    print('JSON entries:', len(df_saved_json))
    print('First JSON entry:')
    print(df_saved_json[0] if df_saved_json else 'No entries found.')